In [3]:
import numpy as np
import pandas as pd
import pymc as pm
import arviz as az

rng = np.random.default_rng(6420)

In [5]:
data_frame = pd.read_csv('data_clean/bfy_age4_clean.csv')

In [9]:
data_frame.shape

(1000, 35)

In [11]:
continuous_vars = ['birthweight_oz', 'gest_age', 'mom_age', 'mom_health', 'cesd', 'n_children', 'n_adults', 'hh_income', 'net_worth']

In [13]:
binary = ['female', 'smoked', 'alcohol', 'biodad_in_hh', 'spanish', 'under4']

In [15]:
# Standardize helper:

def standardize(dataframe):
    continuous = [col for col in dataframe if dataframe[col].nunique() > 2]
    standardized = dataframe.copy()

    standardized[continuous] = (dataframe[continuous] - dataframe[continuous].mean()) / (2 * dataframe[continuous].std())

    return standardized

In [17]:
covariates = ['birthweight_oz', 'gest_age', 'mom_age', 'mom_health', 'cesd', 'n_children', 'n_adults', 'hh_income', 'net_worth', 'female', 'smoked', 'alcohol', 'biodad_in_hh', 'spanish', 'under4']

In [21]:
df_standard = standardize(data_frame[covariates])
covs = df_standard.to_numpy(copy=True)

missing = np.isnan(covs)
covs[np.isnan(covs)] = 0

In [23]:
covs.shape, missing.shape, missing.sum(axis=0)

((1000, 15),
 (1000, 15),
 array([ 24,   5,   0,   0,  10,   0,   0,   0, 111,   0,   0,   0,   0,
        168, 108]))

In [31]:
categorical_vars = ['site', 'mom_educ', 'mom_race', 'marital', 'assessor']

parts = [covs, missing]
for elem in categorical_vars: 
    x = data_frame[elem].to_numpy(copy=True)
    x[np.isnan(x)] = 99
    parts.append(pd.get_dummies(x).to_numpy()[:, 1:])

matrix = np.concatenate(parts, axis=1)

matrix.shape

(1000, 67)

In [33]:
treat = data_frame['treat'].to_numpy(copy=True)
y = data_frame['vocab_z'].to_numpy(copy=True)

missing_mask = np.isnan(y)
y_vocab = y[~missing_mask]
treat_vocab = treat[~missing_mask]
matrix_vocab = matrix[~missing_mask]

varies_mask = matrix_vocab.std(axis=0) > 0
matrix_vocab = matrix_vocab[:, varies_mask]

y_vocab.shape, treat_vocab.shape, matrix_vocab.shape

((700,), (700,), (700, 55))

In [39]:
with pm.Model() as model_vocab:
    tau = pm.Gamma('tau', 0.001, 0.001)
    alpha = pm.Normal('intercept', 0, tau=0.001)
    delta = pm.Normal('delta', 0, tau=0.001)
    beta = pm.Normal('beta', 0, tau = 0.001, shape=matrix_vocab.shape[1])

    variance = pm.Deterministic('variance', 1 / tau)

    mu = alpha + delta * treat_vocab + pm.math.dot(matrix_vocab, beta)

    pm.Normal('likelihood', mu=mu, tau= tau, observed=y_vocab)

    trace = pm.sample(3000, random_seed=6420)


NUTS[nutpie]: [tau, intercept, delta, beta]


/Users/abtinkhashayar/pymc_env/lib/python3.12/site-packages/rich/live.py:260: UserWarning: install "ipywidgets" for
Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

In [41]:
az.summary(trace, var_names='~beta', ci_prob=0.95)

,mean,sd,eti95_lb,eti95_ub,ess_bulk,ess_tail,r_hat,mcse_mean,mcse_sd
intercept,-0.04,0.37,-0.76,0.68,1088,1708,1.00,0.011,0.0066
delta,-0.058,0.07,-0.2,0.08,14567,8522,1.00,0.00058,0.00072
tau,1.232,0.069,1.1,1.4,16620,8126,1.00,0.00054,0.00068
variance,0.8141,0.0457,0.73,0.91,16620,8126,1.00,0.00035,0.00045
